In [ ]:
# ==============================================================================
# UNIFIED ANALYSIS PIML scintillator Light Yield
# ==============================================================================
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from sklearn.model_selection import RandomizedSearchCV
from sklearn.feature_selection import RFECV
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import xgboost as xgb
import shap

warnings.filterwarnings('ignore')

# --- Global Publication Aesthetics & Theme ---
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['DejaVu Sans', 'Arial', 'Helvetica'],
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'font.size': 9.5,
    'axes.labelsize': 10,
    'axes.titlesize': 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'axes.linewidth': 0.9,
    'axes.edgecolor': '#222222',
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'xtick.top': True,
    'ytick.right': True,
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight'
})

# ==============================================================================
# 0. CONFIGURATION & DIRECTORY SETUP
# ==============================================================================
DATA_PATH = '/content/drive/MyDrive/PET_work/all_scintillators.csv'
OUTPUT_DIR = '/content/drive/MyDrive/PET_work/figures_pub/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("--- 1. LOADING AND STANDARDIZING DATASET ---")
try:
    df = pd.read_csv(DATA_PATH)
except FileNotFoundError:
    print(f"Error: Could not find raw file at {DATA_PATH}. Please verify your Drive path.")
    exit()

# Standardize column names to snake_case
df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(' ', '_', regex=False)
    .str.replace('(', '', regex=False)
    .str.replace(')', '', regex=False)
    .str.replace('/', '', regex=False)
)

# ==============================================================================
# 2. RIGOROUS DATA CLEANING & IMPUTATION (N = 295)
# ==============================================================================
print("\n--- 2. DATA CLEANING & PREPROCESSING ---")
df_cleaned = df.copy()
initial_count = len(df_cleaned)
df_cleaned.drop_duplicates(inplace=True)
df_cleaned.dropna(subset=['lightyield'], inplace=True)

# Drop high-missingness columns (~58.6% dielectric constants)
cols_to_drop = ['av_el_diel_cons', 'av_ion_diel_cons']
df_cleaned.drop(columns=[c for c in cols_to_drop if c in df_cleaned.columns], inplace=True)

# Median imputation for remaining features
numeric_cols_imp = df_cleaned.select_dtypes(include=np.number).columns.tolist()
if 'lightyield' in numeric_cols_imp:
    numeric_cols_imp.remove('lightyield')
for col in numeric_cols_imp:
    if df_cleaned[col].isnull().any():
        df_cleaned[col].fillna(df_cleaned[col].median(), inplace=True)

# ==============================================================================
# 3. PHYSICS-INFORMED FEATURE ENGINEERING
# ==============================================================================
print("\n--- 3. PHYSICS-INFORMED FEATURE ENGINEERING ---")
epsilon = 1e-6
df_cleaned['electronegativity_mismatch'] = np.abs(df_cleaned['av.ele+'] - df_cleaned['av.ele-'])
df_cleaned['size_mismatch'] = np.abs(df_cleaned['emp_rad+'] - df_cleaned['emp_rad-']) / (
    df_cleaned['emp_rad+'] + df_cleaned['emp_rad-'] + epsilon
)
df_cleaned['bandgap_doping_ratio'] = df_cleaned['host_band_gap'] / (df_cleaned['dop_conc'] + epsilon)

# ==============================================================================
# 4. MODEL TRAINING, RFECV, & HYPERPARAMETER TUNING
# ==============================================================================
print("\n--- 4. MODEL TRAINING & SELECTION PIPELINE ---")
features_to_exclude = ['lightyield', 'scintillator', 'mechanism', 'reference']
X = df_cleaned.drop(columns=[col for col in features_to_exclude if col in df_cleaned.columns])
X = X.select_dtypes(include=np.number)
y = np.log1p(df_cleaned['lightyield'])

models_dict = {
    'RandomForest': {
        'model': RandomForestRegressor(random_state=42),
        'params': {
            'n_estimators': [100, 200, 300],
            'max_depth': [10, 20, 30, None],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4],
            'max_features': ['sqrt', 'log2', 1.0]
        }
    },
    'GradientBoosting': {
        'model': GradientBoostingRegressor(random_state=42),
        'params': {
            'n_estimators': [100, 200, 300],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth': [3, 5, 7],
            'subsample': [0.8, 0.9, 1.0]
        }
    },
    'XGBoost': {
        'model': xgb.XGBRegressor(random_state=42, objective='reg:squarederror'),
        'params': {
            'n_estimators': [100, 200, 300],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth': [3, 5, 7],
            'subsample': [0.8, 0.9, 1.0],
            'colsample_bytree': [0.8, 0.9, 1.0]
        }
    }
}

results = {}
for name, cfg in models_dict.items():
    print(f"\nProcessing model: {name}")
    selector = RFECV(estimator=cfg['model'], step=1, cv=5, scoring='r2', n_jobs=-1)
    selector.fit(X, y)
    X_sel = X.loc[:, selector.support_]

    rand_search = RandomizedSearchCV(
        estimator=cfg['model'], param_distributions=cfg['params'],
        n_iter=25, cv=5, random_state=42, n_jobs=-1, scoring='r2'
    )
    rand_search.fit(X_sel, y)
    best_est = rand_search.best_estimator_

    y_pred_log = best_est.predict(X_sel)
    y_pred_orig = np.expm1(y_pred_log)
    y_actual_orig = df_cleaned['lightyield'].values

    r2_val = r2_score(y_actual_orig, y_pred_orig)
    rmse_val = np.sqrt(mean_squared_error(y_actual_orig, y_pred_orig))

    results[name] = {
        'best_model': best_est,
        'r2_score': r2_val,
        'rmse': rmse_val,
        'y_pred': y_pred_orig,
        'y_actual': y_actual_orig,
        'band_gap': df_cleaned['host_band_gap'].values,
        'X_selected': X_sel
    }
    print(f"--> {name} Optimized | R2: {r2_val:.3f} | RMSE: {rmse_val:.1f} ph/MeV")

# Identify best performing model dynamically
best_model_name = max(results, key=lambda k: results[k]['r2_score'])
best_data = results[best_model_name]

# ==============================================================================
# 5. AUTOMATED GENERATION OF ALL UNIQUE PUBLICATION FIGURES
# ==============================================================================
print("\n--- 5. GENERATING PUBLICATION-GRADE FIGURES ---")

# --- Figure S1: Feature Distributions ---
canonical_labels = {
    'lightyield': r'Light Yield, $\mathrm{LY}$ (ph/MeV)',
    'dop_conc': r'Dopant Concentration, $c_{\mathrm{dop}}$ (%)',
    'host_band_gap': r'Host Band Gap, $E_g$ (eV)',
    'density_gcc': r'Density, $\rho$ ($\mathrm{g/cm^3}$)',
    'electronegativity_mismatch': r'Electronegativity Mismatch, $\Delta\chi$',
    'size_mismatch': r'Ionic Radius Mismatch, $\Delta r$',
    'bandgap_doping_ratio': r'Ratio $E_g / c_{\mathrm{dop}}$'
}

num_cols = df_cleaned.select_dtypes(include=np.number).columns.tolist()
n_cols_grid = 4
n_rows_grid = int(np.ceil(len(num_cols) / n_cols_grid))
fig, axes = plt.subplots(n_rows_grid, n_cols_grid, figsize=(11.5, 2.5 * n_rows_grid))
axes = np.array(axes).flatten()

for idx, col in enumerate(num_cols):
    ax = axes[idx]
    srs = df_cleaned[col].dropna()
    lbl = canonical_labels.get(col, col.replace('_', ' ').title())
    bins = np.logspace(np.log10(srs.min()), np.log10(srs.max()), 22) if (srs.min() > 0 and srs.max()/srs.min() > 50) else 20
    if srs.min() > 0 and srs.max()/srs.min() > 50:
        ax.set_xscale('log')

    ax.hist(srs, bins=bins, color='#2b5c8f', alpha=0.55, edgecolor='#111111', linewidth=0.75, zorder=2)
    med = srs.median()
    med_str = f"Med: {med:.1f}" if med < 1000 else f"Med: {med:,.0f}"
    ax.axvline(med, color='#b33a00', linestyle='--', linewidth=1.1, label=med_str, zorder=3)
    ax.text(0.04, 0.90, f"({chr(97 + idx)})", transform=ax.transAxes, fontsize=8.5, fontweight='bold', va='top')
    ax.set_title(lbl, fontsize=8.5, fontweight='semibold', pad=5)
    ax.tick_params(labelsize=7.5, direction='in')
    ax.grid(True, linestyle=':', alpha=0.45, zorder=1)
    ax.legend(fontsize=7.0, loc='upper right', framealpha=0.9)

for j in range(len(num_cols), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
for ext in ['pdf', 'png']:
    plt.savefig(os.path.join(OUTPUT_DIR, f'Figure_S1_feature_distributions.{ext}'), format=ext)
plt.close()

# --- Figure S2: Top Scintillator Host Compound Frequency ---
host_col = 'scintillator' if 'scintillator' in df_cleaned.columns else 'Scintillator'
top_hosts = df_cleaned[host_col].value_counts().head(20)
fig, ax = plt.subplots(figsize=(6.5, 5.5))
y_pos = np.arange(len(top_hosts))
bars = ax.barh(y_pos, top_hosts.values, height=0.65, color='#2b5c8f', alpha=0.75, edgecolor='#111111', linewidth=0.8, zorder=3)
ax.invert_yaxis()

for bar in bars:
    w = bar.get_width()
    ax.text(w + 0.35, bar.get_y() + bar.get_height()/2, f"{int(w)}", va='center', ha='left', fontsize=8, fontweight='semibold')

ax.set_yticks(y_pos)
ax.set_yticklabels(top_hosts.index, fontsize=8.5)
ax.set_xlabel('Sample Count in Dataset', fontsize=9.5)
ax.set_title('Top 20 Scintillator Host Matrices by Frequency', fontweight='bold', fontsize=10.5, pad=10)
ax.set_xlim(0, max(top_hosts.values) + 4)
ax.grid(axis='x', linestyle=':', alpha=0.5, zorder=0)
plt.tight_layout()
for ext in ['pdf', 'png']:
    plt.savefig(os.path.join(OUTPUT_DIR, f'Figure_S2_compound_frequency.{ext}'), format=ext)
plt.close()

# --- Main Parity Plots (Vertical Publication Stack) ---
SCALE = 1e3
all_vals = np.concatenate([d['y_actual'] for d in results.values()] + [d['y_pred'] for d in results.values()]) / SCALE
val_min, val_max = 0, np.percentile(all_vals, 99.8) * 1.05
panel_labels = ['(a)', '(b)', '(c)']

fig, axes = plt.subplots(len(results), 1, figsize=(3.5, 3.2 * len(results)), sharex=True)
for i, (model_name, data) in enumerate(results.items()):
    ax = axes[i]
    y_act = data['y_actual'] / SCALE
    y_pr = data['y_pred'] / SCALE
    mae_val = mean_absolute_error(data['y_actual'], data['y_pred'])

    ax.plot([val_min, val_max], [val_min, val_max], color='#d62728', linestyle='--', linewidth=1.1, zorder=2)
    ax.scatter(y_act, y_pr, s=18, alpha=0.7, color='#2b5c8f', edgecolors='white', linewidths=0.3, zorder=3)

    stats_text = f"$R^2 = {data['r2_score']:.3f}$\nRMSE = {data['rmse']:,.0f}\nMAE = {mae_val:,.0f}"
    ax.text(0.95, 0.05, stats_text, transform=ax.transAxes, va='bottom', ha='right', fontsize=8, family='monospace',
            bbox=dict(boxstyle='square,pad=0.35', facecolor='white', edgecolor='#e2e2e2', alpha=0.9))

    ax.set_xlim(val_min, val_max)
    ax.set_ylim(val_min, val_max)
    ax.set_aspect('equal', adjustable='box')
    ax.grid(True, linestyle=':', alpha=0.5, zorder=1)
    ax.xaxis.set_major_locator(ticker.MultipleLocator(25))
    ax.yaxis.set_major_locator(ticker.MultipleLocator(25))
    ax.ticklabel_format(useOffset=False, style='plain')
    ax.set_title(f"{panel_labels[i]} {model_name}", fontweight='bold', fontsize=9.5, pad=8)
    ax.set_ylabel(r'Predicted LY ($10^3$ ph/MeV)', fontsize=8.5)
    if i == len(results) - 1:
        ax.set_xlabel(r'Measured LY ($10^3$ ph/MeV)', fontsize=8.5)

plt.tight_layout()
for ext in ['pdf', 'png']:
    plt.savefig(os.path.join(OUTPUT_DIR, f'parity_vertical_optimized.{ext}'), format=ext, bbox_inches='tight')
plt.close()

# --- Physics Benchmark Figure ---
band_gaps = best_data['band_gap']
y_actual = best_data['y_actual']
y_pred = best_data['y_pred']
ly_max_nominal = 1e6 / (2.5 * band_gaps)
efficiency_exp = y_actual / ly_max_nominal
efficiency_ml = y_pred / ly_max_nominal

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7.5, 3.5))
bg_grid = np.linspace(max(1.0, band_gaps.min() * 0.95), 8.5, 500)
ly_grid_upper = (1e6 / (2.0 * bg_grid)) / SCALE
ly_grid_lower = (1e6 / (3.0 * bg_grid)) / SCALE

ax1.fill_between(bg_grid, ly_grid_lower, ly_grid_upper, color='#e5e5e5', alpha=0.7, zorder=1)
ax1.plot(bg_grid, ly_grid_upper, color='#222222', linestyle='-', lw=1.3, zorder=2)
ax1.plot(bg_grid, ly_grid_lower, color='#555555', linestyle='--', lw=1.3, zorder=2)
ax1.scatter(band_gaps, y_actual / SCALE, s=20, marker='o', facecolor='#2b5c8f', edgecolor='#111111', linewidth=0.5, alpha=0.65, label='Experimental', zorder=3)
ax1.scatter(band_gaps, y_pred / SCALE, s=26, marker='d', facecolor='none', edgecolor='#b33a00', linewidth=1.1, alpha=0.85, label=f'ML ({best_model_name})', zorder=4)
ax1.set_xlim(0.8, 8.5)
ax1.set_ylim(0, 130)
ax1.set_xlabel(r'Host Band Gap, $E_g$ (eV)', fontsize=10)
ax1.set_ylabel(r'Light Yield ($10^3$ ph/MeV)', fontsize=10)
ax1.set_title('(a) Benchmark vs. Fundamental Limit', fontweight='bold', fontsize=10.5, pad=8)
ax1.grid(True, linestyle=':', alpha=0.45)
ax1.legend(frameon=True, fontsize=8, loc='lower left', framealpha=0.92, edgecolor='#cccccc')

bins = np.linspace(0, max(np.percentile(efficiency_exp, 99.5), np.percentile(efficiency_ml, 99.5)) * 1.15, 25)
ax2.hist(efficiency_exp, bins=bins, density=True, histtype='stepfilled', facecolor='#2b5c8f', alpha=0.25, hatch='///', edgecolor='#2b5c8f', lw=1.2, label=r'Exp. $\eta_{\mathrm{exp}}$')
ax2.hist(efficiency_ml, bins=bins, density=True, histtype='stepfilled', facecolor='#b33a00', alpha=0.15, hatch=r'\\\\', edgecolor='#b33a00', lw=1.2, label=r'ML $\eta_{\mathrm{ML}}$')
ax2.set_xlabel(r'Efficiency Ratio, $\eta$ ($\mathrm{LY} / \mathrm{LY}_{\mathrm{max,nominal}}$)', fontsize=10)
ax2.set_ylabel('Probability Density', fontsize=10)
ax2.set_title('(b) Scintillation Efficiency Distribution', fontweight='bold', fontsize=10.5, pad=8)
ax2.grid(True, linestyle=':', alpha=0.45)
ax2.legend(frameon=True, fontsize=8, loc='upper right', framealpha=0.92, edgecolor='#cccccc')

plt.tight_layout()
for ext in ['pdf', 'png']:
    plt.savefig(os.path.join(OUTPUT_DIR, f'physics_benchmark_publication.{ext}'), format=ext)
plt.close()

# --- SHAP Summary Plots (Figures S7a & S7b) ---
X_sel_shap = best_data['X_selected'].copy()
feature_name_map = {
    'host_band_gap': r'Host Band Gap, $E_g$',
    'dop_conc': 'Dopant Concentration',
    'density_gcc': r'Density ($\mathrm{g/cm^3}$)',
    'av_ion_pot': 'Mean Ionization Potential',
    'ion_pot+': r'Cation Ionization Pot. ($I^+$)',
    'ion_pot-': r'Anion Ionization Pot. ($I^-$)',
    'av_electroneg': r'Mean Electronegativity ($\bar{\chi}$)',
    'av.ele+': r'Cation Electronegativity ($\chi^+$)',
    'av.ele-': r'Anion Electronegativity ($\chi^-$)',
    'av_elec_aff': 'Mean Electron Affinity',
    'elec_aff+': r'Cation Electron Affinity ($EA^+$)',
    'elec_aff-': r'Anion Electron Affinity ($EA^-$)',
    'emp_rad+': r'Cation Empirical Radius ($r^+$)',
    'emp_rad-': r'Anion Empirical Radius ($r^-$)',
    'electronegativity_mismatch': r'$\Delta\chi$ Mismatch',
    'size_mismatch': r'Ionic Radius Mismatch ($\Delta r$)',
    'bandgap_doping_ratio': r'Ratio $E_g / \mathrm{Conc_{dop}}$'
}
X_sel_shap.rename(columns=lambda c: feature_name_map.get(c, c), inplace=True)

explainer = shap.TreeExplainer(best_data['best_model'])
shap_values = explainer.shap_values(X_sel_shap)

# Figure S7a: Global Importance Bar Plot
fig, ax = plt.subplots(figsize=(4.8, 4.2))
shap.summary_plot(shap_values, X_sel_shap, plot_type="bar", color='#2b5c8f', max_display=10, show=False)
plt.title('Global Feature Importance', fontweight='bold', fontsize=11, pad=10)
plt.xlabel(r'Mean (|SHAP Value|) on $\ln(1 + \mathrm{LY})$', fontsize=9.5)
plt.grid(axis='x', linestyle=':', alpha=0.5, zorder=0)
plt.tight_layout()
for ext in ['pdf', 'png']:
    plt.savefig(os.path.join(OUTPUT_DIR, f'Figure_S7a_feature_importance.{ext}'), format=ext, bbox_inches='tight')
plt.close()

# Figure S7b: Directional Beeswarm Plot
fig, ax = plt.subplots(figsize=(5.4, 4.2))
shap.summary_plot(shap_values, X_sel_shap, max_display=10, cmap='coolwarm', alpha=0.65, show=False)
plt.title('Directional SHAP Value Distribution', fontweight='bold', fontsize=11, pad=10)
plt.xlabel(r'SHAP Value (Impact on $\ln(1 + \mathrm{LY})$)', fontsize=9.5)
plt.grid(True, linestyle=':', alpha=0.35, zorder=0)
plt.tight_layout()
for ext in ['pdf', 'png']:
    plt.savefig(os.path.join(OUTPUT_DIR, f'Figure_S7b_shap_beeswarm.{ext}'), format=ext, bbox_inches='tight')
plt.close()

print(f"\nPipeline successfully completed! All models trained and publication figures rendered to: {OUTPUT_DIR}")

--- 1. LOADING AND STANDARDIZING DATASET ---

--- 2. DATA CLEANING & PREPROCESSING ---

--- 3. PHYSICS-INFORMED FEATURE ENGINEERING ---

--- 4. MODEL TRAINING & SELECTION PIPELINE ---

Processing model: RandomForest
--> RandomForest Optimized | R2: 0.816 | RMSE: 11023.1 ph/MeV

Processing model: GradientBoosting
--> GradientBoosting Optimized | R2: 0.969 | RMSE: 4523.6 ph/MeV

Processing model: XGBoost
--> XGBoost Optimized | R2: 0.549 | RMSE: 17263.3 ph/MeV

--- 5. GENERATING PUBLICATION-GRADE FIGURES ---

Pipeline successfully completed! All models trained and publication figures rendered to: /content/drive/MyDrive/PET_work/figures_pub/
